# Milestone 2 RAG Exploration

This notebook explores the Milestone 2 backend workflow for the Amazon Reviews 2023 **Video_Games** project.

It includes:

- LLM/API setup and sanity checks
- semantic retrieval + semantic RAG
- hybrid retrieval + hybrid RAG
- prompt variant comparison
- backend tests that can replace terminal testing
- saved example outputs for later discussion and UI handoff

This notebook assumes:
- raw data and processed Milestone 1 sample artifacts already exist
- a local `.env` file exists in the project root
- the `.env` file contains:
  - `GROQ_API_KEY`
  - `LLM_MODEL` (for example `llama-3.1-8b-instant`)


## Workflow diagram

```mermaid
flowchart TD
    A[User Query] --> B[Retriever]
    B --> C[Top-k Documents]
    C --> D[Context Builder]
    D --> E[Prompt Template]
    E --> F[Hosted LLM API]
    F --> G[Generated Answer]

    subgraph Semantic RAG
        B1[Semantic Retriever / FAISS]
    end

    subgraph Hybrid RAG
        B2[BM25 Retriever]
        B3[Semantic Retriever]
        B4[Hybrid Merger / RRF]
    end

    A --> B1
    B1 --> C

    A --> B2
    A --> B3
    B2 --> B4
    B3 --> B4
    B4 --> C
```


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

import os
import pandas as pd
from dotenv import load_dotenv

load_dotenv(dotenv_path=PROJECT_ROOT / ".env")

from langchain_groq import ChatGroq

from src.semantic import SemanticRetriever
from src.bm25 import BM25Retriever
from src.hybrid import HybridRetriever
from src.rag_pipeline import SemanticRAGPipeline, HybridRAGPipeline

## Paths and configuration

In [ ]:
processed_dir = PROJECT_ROOT / "data" / "processed"

CORPUS_PATH = processed_dir / "video_games_corpus_sample.parquet"
BM25_INDEX_PATH = processed_dir / "bm25_sample_index.pkl"
BM25_TOKENS_PATH = processed_dir / "bm25_sample_tokens.pkl"
FAISS_INDEX_PATH = processed_dir / "faiss_sample.index"
SEMANTIC_METADATA_PATH = processed_dir / "semantic_sample_metadata.pkl"

LLM_MODEL = os.getenv("LLM_MODEL", "llama-3.1-8b-instant")

print("Processed dir:", processed_dir)
print("LLM model:", LLM_MODEL)

## Check that Milestone 1 artifacts exist

In [ ]:
required_files = [
    CORPUS_PATH,
    BM25_INDEX_PATH,
    BM25_TOKENS_PATH,
    FAISS_INDEX_PATH,
    SEMANTIC_METADATA_PATH,
]

for path in required_files:
    print(path.name, "->", path.exists())

## API key sanity check

In [ ]:
print("Has GROQ_API_KEY:", bool(os.getenv("GROQ_API_KEY")))
print("Model from env:", os.getenv("LLM_MODEL"))

## Test the Groq API directly

In [ ]:
llm = ChatGroq(
    model=LLM_MODEL,
    api_key=os.getenv("GROQ_API_KEY"),
)

response = llm.invoke("Say hello in one short sentence.")
print(response.content)

## Model choice and rationale

For Milestone 2, we use a hosted LLM through the Groq API instead of running a local model.
This is useful because:

- it avoids local model downloads and hardware constraints
- it keeps setup lighter for a laptop-based project
- it lets us focus on retrieval, prompts, and grounded generation

The current model is controlled by the `.env` variable `LLM_MODEL`.


## Test the semantic retriever only

In [ ]:
semantic_retriever = SemanticRetriever.load_saved(
    index_path=FAISS_INDEX_PATH,
    metadata_path=SEMANTIC_METADATA_PATH,
)

semantic_docs = semantic_retriever.search(
    "story-rich scary game with dark atmosphere",
    top_k=5,
)

semantic_docs[["product_title", "review_title", "rating", "semantic_score"]].head()

## Semantic RAG pipeline

In [ ]:
semantic_rag = SemanticRAGPipeline(
    corpus_path=str(CORPUS_PATH),
    faiss_index_path=str(FAISS_INDEX_PATH),
    metadata_path=str(SEMANTIC_METADATA_PATH),
    model_name=LLM_MODEL,
    top_k=5,
)

### Test semantic RAG with one query

In [ ]:
semantic_result = semantic_rag.answer("story-rich scary game with dark atmosphere")

print("=== SEMANTIC RAG ANSWER ===")
print(semantic_result["answer"])

In [ ]:
semantic_result["docs"][["product_title", "review_title", "rating"]].head()

In [ ]:
print("=== CONTEXT PREVIEW ===")
print(semantic_result["context"][:1500])

## Prompt variants

In [ ]:
SYSTEM_PROMPT_V1 = """
You are a helpful Amazon shopping assistant.
Answer the question using ONLY the provided context from Amazon product reviews and metadata.
Do not make up product details that are not in the context.
When possible, mention the product title or ASIN.
"""

SYSTEM_PROMPT_V2 = """
You are a careful product recommendation assistant.
Use only the retrieved Amazon review context below.
If the context is insufficient, say so clearly.
Keep the answer concise, helpful, and grounded in the retrieved evidence.
"""

SYSTEM_PROMPT_V3 = """
You are an Amazon reviews analyst.
Answer the user only from the retrieved reviews and product metadata.
Prefer direct evidence from the retrieved context, and avoid unsupported claims.
"""

In [ ]:
semantic_rag_v1 = SemanticRAGPipeline(
    corpus_path=str(CORPUS_PATH),
    faiss_index_path=str(FAISS_INDEX_PATH),
    metadata_path=str(SEMANTIC_METADATA_PATH),
    model_name=LLM_MODEL,
    top_k=5,
    system_prompt=SYSTEM_PROMPT_V1,
)

semantic_rag_v2 = SemanticRAGPipeline(
    corpus_path=str(CORPUS_PATH),
    faiss_index_path=str(FAISS_INDEX_PATH),
    metadata_path=str(SEMANTIC_METADATA_PATH),
    model_name=LLM_MODEL,
    top_k=5,
    system_prompt=SYSTEM_PROMPT_V2,
)

semantic_rag_v3 = SemanticRAGPipeline(
    corpus_path=str(CORPUS_PATH),
    faiss_index_path=str(FAISS_INDEX_PATH),
    metadata_path=str(SEMANTIC_METADATA_PATH),
    model_name=LLM_MODEL,
    top_k=5,
    system_prompt=SYSTEM_PROMPT_V3,
)

In [ ]:
test_query = "best racing game with fun tracks"

ans_v1 = semantic_rag_v1.answer(test_query)["answer"]
ans_v2 = semantic_rag_v2.answer(test_query)["answer"]
ans_v3 = semantic_rag_v3.answer(test_query)["answer"]

print("=== Prompt V1 ===")
print(ans_v1)
print("\n=== Prompt V2 ===")
print(ans_v2)
print("\n=== Prompt V3 ===")
print(ans_v3)

### Prompt observation notes

Add short notes here after comparing the outputs:

- Which prompt gave the most grounded answer?
- Which prompt was the most concise?
- Which prompt avoided unsupported claims best?


## Test the hybrid retriever only

In [ ]:
bm25_retriever = BM25Retriever.load_saved(
    corpus_path=CORPUS_PATH,
    index_path=BM25_INDEX_PATH,
    tokens_path=BM25_TOKENS_PATH,
)

semantic_retriever = SemanticRetriever.load_saved(
    index_path=FAISS_INDEX_PATH,
    metadata_path=SEMANTIC_METADATA_PATH,
)

hybrid_retriever = HybridRetriever(
    bm25_retriever=bm25_retriever,
    semantic_retriever=semantic_retriever,
    top_k=5,
)

hybrid_docs = hybrid_retriever.search(
    "story-rich scary game with dark atmosphere",
    top_k=5,
)

hybrid_docs[["product_title", "review_title", "rating", "hybrid_score"]].head()

## Hybrid RAG pipeline

In [ ]:
hybrid_rag = HybridRAGPipeline(
    corpus_path=str(CORPUS_PATH),
    bm25_index_path=str(BM25_INDEX_PATH),
    bm25_tokens_path=str(BM25_TOKENS_PATH),
    faiss_index_path=str(FAISS_INDEX_PATH),
    metadata_path=str(SEMANTIC_METADATA_PATH),
    model_name=LLM_MODEL,
    top_k=5,
)

In [ ]:
hybrid_result = hybrid_rag.answer("story-rich scary game with dark atmosphere")

print("=== HYBRID RAG ANSWER ===")
print(hybrid_result["answer"])

In [ ]:
hybrid_result["docs"][["product_title", "review_title", "rating"]].head()

## Compare semantic RAG vs hybrid RAG

In [ ]:
comparison_query = "story-rich scary game with dark atmosphere"

semantic_answer = semantic_rag.answer(comparison_query)["answer"]
hybrid_answer = hybrid_rag.answer(comparison_query)["answer"]

print("=== SEMANTIC RAG ===")
print(semantic_answer)
print("\n=== HYBRID RAG ===")
print(hybrid_answer)

### Comparison notes

Add short notes here after testing:

- Did hybrid retrieval produce more relevant supporting documents?
- Did the hybrid answer look more complete?
- Was one answer more grounded than the other?


## Save example outputs

In [ ]:
semantic_result["docs"].to_csv(processed_dir / "milestone2_semantic_docs_preview.csv", index=False)
hybrid_result["docs"].to_csv(processed_dir / "milestone2_hybrid_docs_preview.csv", index=False)

with open(processed_dir / "milestone2_semantic_answer.txt", "w", encoding="utf-8") as f:
    f.write(semantic_result["answer"])

with open(processed_dir / "milestone2_hybrid_answer.txt", "w", encoding="utf-8") as f:
    f.write(hybrid_result["answer"])

print("Saved example outputs to data/processed/")

## Reproducibility test checklist

Before handing this backend to a teammate for UI work, confirm all of these:

- `.env` loads correctly
- Groq API call works
- semantic retriever loads and returns docs
- semantic RAG returns an answer
- hybrid retriever loads and returns docs
- hybrid RAG returns an answer
- outputs can be saved under `data/processed/`

If all of these pass, the Milestone 2 backend is in good shape for handoff.
